In [1]:
import pandas as pd
import numpy as np

In [2]:
format_counts = pd.read_csv("Processed Data/format_counts.csv")
plaintexts = pd.read_csv("Processed Data/cleaned_data.csv")
formats = pd.read_csv("Processed Data/formats.csv")

C:\Users\MarkusLarsen\AppData\Local\Temp\ipykernel_9948\872953096.py:1: DtypeWarning: Columns (0,2,4,10,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  format_counts = pd.read_csv("Processed Data/format_counts.csv")
C:\Users\MarkusLarsen\AppData\Local\Temp\ipykernel_9948\872953096.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  plaintexts = pd.read_csv("Processed Data/cleaned_data.csv")


Min-entropy for the plaintexts:

In [3]:
h_inf = {}
for col in plaintexts.columns:
    probs = plaintexts[col].value_counts(normalize=True)
    max_prob = probs.max()
    h_inf_value = -np.log2(max_prob)
    h_inf[col] = h_inf_value

Conditional min-entropy:

In [4]:
h_inf_cond = {}
for col in formats.columns:
    joint = pd.DataFrame({"x": plaintexts[col], "f": formats[col]}) # Create a joint DataFrame of plaintext and format columns
    joint = joint.replace("", pd.NA).dropna()

    p_f = joint["f"].value_counts(normalize=True) # Probability distribution of the format column
    # Maximum probability of x given each format value
    max_p_x_given_f = (
        joint.groupby("f")["x"]
        .value_counts(normalize=True)
        .groupby(level=0)
        .max()
    )
    vulnerability = (p_f * max_p_x_given_f).sum() # Overall vulnerability of the format column
    h_inf_cond_value = -np.log2(vulnerability)

    h_inf_cond[col] = h_inf_cond_value

Count the total number of formats (including distinct and unique formats) for each attribute and calculate leakage using both Shannon and min-entropy.

In [5]:
for col in format_counts.columns:
    if col.endswith("_count"):
        continue
    count_series = format_counts[f"{col}_count"].dropna()
    count_values = count_series.to_numpy(dtype=float)
    total_count = count_values.sum()
    distinct_count = len(count_values)
    unique_count = count_series.eq(1).sum()
    pct = (unique_count / total_count * 100) if total_count > 0 else 0.0
    shannon_entropy = -np.sum((count_values / total_count) * np.log2(count_values / total_count))
    min_entropy = h_inf[col] - h_inf_cond[col]
    print(f"{col}: Total = {total_count}, Distinct = {distinct_count}, Unique Count = {unique_count}, Pct = {pct:.2f}%, Shannon = {shannon_entropy:.2f}, Min = {min_entropy:.2f}")

NAME: Total = 2930166.0, Distinct = 15037, Unique Count = 6549, Pct = 0.22%, Shannon = 7.36, Min = 7.03
CITY: Total = 2929104.0, Distinct = 1095, Unique Count = 412, Pct = 0.01%, Shannon = 4.64, Min = 2.98
ZIP_CODE: Total = 2924730.0, Distinct = 5, Unique Count = 0, Pct = 0.00%, Shannon = 0.92, Min = 0.62
EMPLOYER: Total = 2664125.0, Distinct = 49107, Unique Count = 23520, Pct = 0.88%, Shannon = 7.08, Min = 1.69
OCCUPATION: Total = 2795679.0, Distinct = 50571, Unique Count = 25958, Pct = 0.93%, Shannon = 5.54, Min = 1.51
FIRST_NAME: Total = 2857117.0, Distinct = 20, Unique Count = 1, Pct = 0.00%, Shannon = 2.62, Min = 2.14
LAST_NAME: Total = 2857116.0, Distinct = 23, Unique Count = 0, Pct = 0.00%, Shannon = 2.85, Min = 1.87
CLEAN_NAME: Total = 2857117.0, Distinct = 299, Unique Count = 28, Pct = 0.00%, Shannon = 5.47, Min = 4.55


Combining the NAME and CITY attributes.

In [6]:
joint_pair = pd.DataFrame({
    "x_name": plaintexts["NAME"],
    "x_city": plaintexts["CITY"],
    "f_name": formats["NAME"],
    "f_city": formats["CITY"],
}).replace("", pd.NA).dropna()

joint_pair["x_pair"] = list(zip(joint_pair["x_name"], joint_pair["x_city"]))
joint_pair["f_pair"] = list(zip(joint_pair["f_name"], joint_pair["f_city"]))

plaintext_probs = joint_pair["x_pair"].value_counts(normalize=True)
h_inf_pair = -np.log2(plaintext_probs.max())

format_probs = joint_pair["f_pair"].value_counts(normalize=True)
max_p_x_given_f = (
    joint_pair.groupby("f_pair")["x_pair"]
    .value_counts(normalize=True)
    .groupby(level=0)
    .max()
    .reindex(format_probs.index)
    .fillna(0.0)
    .to_numpy()
    )
vulnerability = (format_probs.to_numpy() * max_p_x_given_f).sum()
h_inf_cond_pair = -np.log2(vulnerability)
min_leakage_pair = h_inf_pair - h_inf_cond_pair

format_counts_pair = joint_pair["f_pair"].value_counts()
distinct_count = len(format_counts_pair)
unique_count = format_counts_pair.eq(1).sum()
total_count = len(joint_pair)
shannon_entropy = -(format_probs * np.log2(format_probs)).sum()

print(
    f"{'NAME + CITY'}: Total = {total_count}, Distinct = {distinct_count}, "
    f"Unique Count = {unique_count}, Pct = {unique_count / total_count * 100:.2f}%, "
    f"Shannon = {shannon_entropy:.2f}, Min = {min_leakage_pair:.2f}"
)

NAME + CITY: Total = 2929104, Distinct = 107303, Unique Count = 50546, Pct = 1.73%, Shannon = 11.89, Min = 10.03
